# Minting and Instance Workflow (Hands-on)

This notebook shows how to generate new BattINFO resources using scripts:
- create a cell-type from datasheet extraction
- create a cell-instance from that type

Outputs are written to a temporary folder and then removed.


In [ ]:
from pathlib import Path
import json
import re
import sys
import subprocess

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    # If notebook is opened from inside notebooks/, move to repo root.
    ROOT = ROOT.parent

assert (ROOT / 'src').exists(), f'Repo root not found from {Path.cwd()}'
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

print('ROOT:', ROOT)


In [ ]:
tmp = ROOT / 'tmp_notebook_workflow'
tmp.mkdir(parents=True, exist_ok=True)
print('tmp dir:', tmp)


In [ ]:
datasheet = ROOT / 'assets' / 'examples' / 'cells' / 'A123__ANR26650M1-B.datasheet.json'
out_cell_type = tmp / 'generated-cell-type.json'

cmd = [
    sys.executable,
    'scripts/normalize_cell_type.py',
    '--datasheet', str(datasheet),
    '--out', str(out_cell_type),
]
res = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
print('exit:', res.returncode)
print(res.stdout)
print(res.stderr)
print('created:', out_cell_type.exists())


In [ ]:
out_cell_instance = tmp / 'generated-cell-instance.json'
cmd = [
    sys.executable,
    'scripts/create_cell_instance.py',
    '--cell-type', str(out_cell_type),
    '--serial', 'LAB-0001',
    '--dataset-id', 'https://w3id.org/battinfo/dataset/1f8r-6v2k-9p4m-3t7x',
    '--out', str(out_cell_instance),
]
res = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
print('exit:', res.returncode)
print(res.stdout)
print(res.stderr)
print('created:', out_cell_instance.exists())


In [ ]:
if out_cell_type.exists():
    doc = json.loads(out_cell_type.read_text(encoding='utf-8'))
    print('cell_type.id:', doc['cell_type']['id'])
if out_cell_instance.exists():
    doc = json.loads(out_cell_instance.read_text(encoding='utf-8'))
    print('cell_instance.id:', doc['cell_instance']['id'])
    print('cell_instance.type_id:', doc['cell_instance']['type_id'])


In [ ]:
# Clean up temporary files
for f in tmp.glob('*'):
    f.unlink()
tmp.rmdir()
print('cleaned')
